# MoS Verification Protocol Demo
Classical verification of quantum learning via Mixture-of-Superpositions,
following [Caro et al. (2023)](https://arxiv.org/abs/2306.04843).

In [6]:
import numpy as np
from mos.mos_simulator import MoSSimulator

from ql.verifier import MoSProtocol, MoSVerifier, ClassicalExampleOracle

## 1. Setup: noisy parity (LPN instance)
Target parity $s^* = 1011_2 = 11$, noise rate $\eta = 0.15$.

In [7]:
n, target_s, eta = 4, 0b1011, 0.15
phi = np.array([(1-eta)*(bin(target_s&x).count('1')%2) + eta*(1-bin(target_s&x).count('1')%2) for x in range(2**n)])
sim = MoSSimulator(n=n, phi=phi, seed=42)

print("True Fourier spectrum (nonzero):")
for s in range(2**n):
    fc = sim.fourier_coefficient(s)
    if fc**2 > 0.01:
        print(f"  s={s:>2} ({s:04b}): phi_hat(s)={fc:+.3f}  |phi_hat(s)|^2={fc**2:.3f}")

True Fourier spectrum (nonzero):
  s=11 (1011): phi_hat(s)=-0.700  |phi_hat(s)|^2=0.490


## 2. Full protocol: prover → verifier → hypothesis

In [8]:
protocol = MoSProtocol(simulator=sim, phi=phi, seed=100)
transcript = protocol.run(epsilon=0.1, delta=0.05, prover_copies=60_000, verifier_samples_per_coeff=20_000)
print(transcript.summary())

PROTOCOL TRANSCRIPT
Parameters: n=4, epsilon=0.1, delta=0.05, theta=0.0025000000000000005

--- Round 1: Prover -> Verifier ---
  Prover sends list L of 2 heavy coefficients: [5, 11]
  Prover's weight estimates:
    s=5: |φ̂(s)|² ≈ 0.002501
    s=11: |φ̂(s)|² ≈ 0.488217

--- Round 2: Verifier checks ---
Verifier Result: ACCEPT
  Reason: Estimated list weight 0.484815 (± 0.121670) is consistent with expected interval [0.440000, 0.540000].
  Classical samples used: 40000
  epsilon=0.1, delta=0.05
  Estimated list weight:  0.484815
  Expected interval:      [0.440000, 0.540000]
  Fourier estimates (2):
         s         bits      φ̂(s)   |φ̂(s)|²    ±stderr
    ------ ------------ ---------- ---------- ----------
        11         1011  +0.696200   0.484694   0.005076
         5         0101  +0.011000   0.000121   0.007071

--- Hypothesis ---
  h(x) = sign(sum_s c_s * chi_s(x)), thresholded to {0,1}
  Coefficients:
    s=5: c_s = +0.011000
    s=11: c_s = +0.696200


In [9]:
ev = protocol.evaluate_hypothesis(transcript, num_samples=50_000)

if ev is None:
    raise RuntimeError("Unexpected hypothesis evaluation to none")

print(f"Hypothesis risk:  {ev['risk']:.4f}")
print(f"Best parity risk: {ev['best_parity_risk']:.4f} (s={ev['best_parity_s']})")
print(f"Excess risk:      {ev['excess_risk']:.4f}")
print(f"Bound met:        {ev['agnostic_bound_met']}")

Hypothesis risk:  0.1513
Best parity risk: 0.1500 (s=11)
Excess risk:      0.0013
Bound met:        True


## 3. Soundness: dishonest prover is rejected

In [10]:
# Two heavy coefficients at s=0 and s=5
n2 = 3
phi2 = np.array([np.clip(0.5 + 0.3*(1-2*(bin(0&x).count('1')%2)) + 0.2*(1-2*(bin(5&x).count('1')%2)), 0, 1) for x in range(8)])
tw = float(np.mean((2*phi2 - 1)**2))

verifier = MoSVerifier(ClassicalExampleOracle(n2, phi2, seed=200), n2)
result = verifier.verify(prover_list=[0], weight_interval=(tw-0.02, tw+0.02),
                         epsilon=0.05, samples_per_coefficient=30_000)

print(f"Dishonest prover sends only s=0, omitting s=5")
print(f"Decision: {result.decision.value}")
print(f"Reason:   {result.reason}")

Dishonest prover sends only s=0, omitting s=5
Decision: REJECT
Reason:   Estimated list weight 0.367398 (± 0.066701) falls below expected lower bound 0.500000. Prover likely omitted heavy coefficients.
